In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, mapping
from libpysal.weights import KNN
from spreg import ML_Lag
import seaborn as sns
import statsmodels.api as sm
import numpy as np
import matplotlib.pyplot as plt
import folium
import seaborn as sns
from esda import Moran

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
property = gpd.read_file('data/cycle/Market_value_usable_hedonic_data.csv', engine='pyogrio')
canopy = gpd.read_file('data/canopy.gdb', engine='pyogrio')
ta = gpd.read_file('data/chc-boundaries/territorial-authority-2021-generalised.gpkg', engine='pyogrio')
sa2 = gpd.read_file('data/chc-boundaries/sa2/statistical-area-2-2023-generalised.shp')

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


In [3]:
pd.set_option('display.max_columns', None)

In [4]:
ta_chc = ta[ta['TA2021_V1_00_NAME_ASCII'] == 'Christchurch City']
sa2_chc = gpd.clip(sa2, ta_chc)
boundary = sa2_chc.copy(deep=True)

In [5]:
# comment this cell to display folium plot
# boundary = sa2_chc[sa2_chc['SA22023__2'].str.lower().str.contains('fendalton')]
canopy = gpd.clip(canopy, boundary)

In [6]:
property_2018 = property[(property['YearSold'].isin(['2017', '2018', '2019'])) & (property['LandArea'] != '0.0')]

# add year dummies
year_dummies = pd.get_dummies(property_2018['YearSold'], prefix='year', drop_first=True)
property_2018_with_dummies = pd.concat([property_2018, year_dummies], axis=1)

In [7]:
property_gdf = gpd.GeoDataFrame(
  property_2018_with_dummies,
  geometry = gpd.points_from_xy(property_2018_with_dummies['gd2000co'], property_2018_with_dummies['gd2000_yco']),
  crs=4326
).to_crs(2193)

gdf_chc = gpd.clip(property_gdf, boundary)
canopy_chc = canopy.copy(deep=True)

In [8]:
gdf_chc.head()

,field_1,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,FullStreetNumber,RoadName,suburb_loc,YearSold,AgeAtSale,GrossSalePrice,CapitalValue,LandValue,ImprovementsValue,PriceValueRelationship,LandArea,TotalFloorArea,gd2000co,gd2000_yco,cycleways_DIST,cycle_DENS,on_DIST,on_DENS,Green_DIST,water_DIST,bus_DIST,Census_Pop,RnkIMDNoEm,RnkIMDNoIn,RnkIMDNoCr,RnkIMDNoHo,RnkIMDNoHe,RnkIMDNoEd,RnkIMDNoAc,DECILE_high,DECILE_prim,Median_Income,CBD_DIST,Ratio,z_score,year_2018,year_2019,geometry
1560,1560,1920,1920,1920,17,Sovereign Gardens,Halswell,2019,24,610000,620000,270000,350000,Market - Index,685.0,211.0,172.56546097,-43.58472623,590.0,1.6,338.1497192,2298.6,98.3508722376,189.6017027471,216.7989101785,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6354.29138933796,1.0163934426229508,0.0487108391601353,False,True,POINT (1564919.129 5174157.305)
3842,3842,4499,4499,4499,8,Sabys Road,Halswell,2017,42,477500,480000,210000,270000,Market - Index,609.0,120.0,172.56705762,-43.58453917,471.0710449,1.6,228.8069763,2303.7000000000003,183.5672524142,139.5812001639,154.8247091108,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6227.39740786588,1.0052356020942408,0.2995445286298665,False,False,POINT (1565047.92 5174178.753)
3671,3671,4298,4298,4298,19,Sovereign Gardens,Halswell,2018,22,655000,630000,255000,375000,Market - Index,704.0,245.0,172.56528655,-43.5845254,610.8361206,1.6,358.6319275,2288.1,72.33949457610001,201.9424953009,192.9410290899,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6352.89184977953,0.9618320610687024,0.1669862470888152,True,False,POINT (1564904.931 5174179.536)
3239,3239,3806,3806,3806,6,Sabys Road,Halswell,2019,44,500000,510000,230000,280000,Market - Index,869.0,131.0,172.56682708,-43.5843312,483.0162354,1.6,233.3314667,2296.0,159.2910754949,157.8540143342,179.6454979671,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6229.0657185615,1.02,0.2404068246655266,False,True,POINT (1565029.188 5174201.753)
3781,3781,4425,4425,4425,20,Sovereign Gardens,Halswell,2018,22,880000,780000,285000,495000,Market - Index,1149.0,279.0,172.56631983,-43.58423973,533.9364624,1.6,273.526001,2291.7,153.0932170892,199.6932187682,216.7306001091,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6258.81186920125,0.8863636363636364,0.7583632867322148,True,False,POINT (1564988.184 5174211.698)


In [9]:
gdf_chc.columns

Index(['field_1', 'Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0',
       'FullStreetNumber', 'RoadName', 'suburb_loc', 'YearSold', 'AgeAtSale',
       'GrossSalePrice', 'CapitalValue', 'LandValue', 'ImprovementsValue',
       'PriceValueRelationship', 'LandArea', 'TotalFloorArea', 'gd2000co',
       'gd2000_yco', 'cycleways_DIST', 'cycle_DENS', 'on_DIST', 'on_DENS',
       'Green_DIST', 'water_DIST', 'bus_DIST', 'Census_Pop', 'RnkIMDNoEm',
       'RnkIMDNoIn', 'RnkIMDNoCr', 'RnkIMDNoHo', 'RnkIMDNoHe', 'RnkIMDNoEd',
       'RnkIMDNoAc', 'DECILE_high', 'DECILE_prim', 'Median_Income', 'CBD_DIST',
       'Ratio', 'z_score', 'year_2018', 'year_2019', 'geometry'],
      dtype='object')

In [10]:
# gdf_chc = gdf_chc.loc[:, ['GrossSalePrice','AgeAtSale', 'LandArea', 'TotalFloorArea', 'water_DIST', 'bus_DIST', 'Census_Pop', 'RnkIMDNoEm',
#        'RnkIMDNoIn', 'RnkIMDNoCr', 'RnkIMDNoHo', 'RnkIMDNoHe', 'RnkIMDNoEd',
#        'RnkIMDNoAc', 'CBD_DIST', 'geometry']]

gdf_chc = gdf_chc.loc[:, ['GrossSalePrice','AgeAtSale', 'LandArea', 'TotalFloorArea', 'water_DIST', 'bus_DIST', 'Census_Pop',
                          'RnkIMDNoEm',
                          'RnkIMDNoIn',
                          'RnkIMDNoCr',
                          'RnkIMDNoHo',
                          'RnkIMDNoHe',
                          'RnkIMDNoEd',
                          'RnkIMDNoAc',
                          'DECILE_high', 'DECILE_prim', 'Median_Income', 'CBD_DIST', 'geometry',
                            'cycleways_DIST', 'cycle_DENS', 
                            'year_2018', 'year_2019',
                            'on_DIST', 'on_DENS'
                     ]]

In [11]:
"""
Optimized Buffer Ring Canopy Density Calculation
Using spatial index intersection method (faster than overlay/sjoin)
Calculates canopy density (%) for rings: 0-20m, 20-50m, 50-100m, 100-200m
"""

import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

# ============================================================================
# STEP 1: CREATE BUFFER RINGS
# ============================================================================

print("Creating buffer rings...")

# Create circular buffers
gdf_chc["buf_20"]  = gdf_chc.geometry.buffer(20)
gdf_chc["buf_50"]  = gdf_chc.geometry.buffer(50)
gdf_chc["buf_100"] = gdf_chc.geometry.buffer(100)
gdf_chc["buf_200"] = gdf_chc.geometry.buffer(200)

print("✓ Buffers created")

# ============================================================================
# STEP 2: BUILD SPATIAL INDEX FOR CANOPY
# ============================================================================

print("\nBuilding spatial index for canopy layer...")
canopy_sindex = canopy_chc.sindex
print("✓ Spatial index built")

# ============================================================================
# STEP 3: CALCULATE CANOPY AREA USING SPATIAL INDEX INTERSECTION
# ============================================================================

def calculate_canopy_in_buffer(property_geom, buffer_geom, canopy_gdf, canopy_sindex):
    """
    Calculate total canopy area within a buffer using spatial index intersection.
    Faster than overlay/sjoin for large datasets.
    
    Parameters:
    -----------
    property_geom : shapely geometry
        Original property point/polygon
    buffer_geom : shapely geometry
        Buffer polygon around property
    canopy_gdf : GeoDataFrame
        Canopy polygons
    canopy_sindex : spatial index
        Pre-built spatial index for canopy
        
    Returns:
    --------
    float : Total canopy area within buffer (m²)
    """
    if buffer_geom.is_empty:
        return 0.0
    
    # Use spatial index to find potential matches
    possible_matches_idx = list(canopy_sindex.intersection(buffer_geom.bounds))
    
    if not possible_matches_idx:
        return 0.0
    
    # Get canopy polygons that might intersect
    possible_matches = canopy_gdf.iloc[possible_matches_idx]
    
    # Filter to actual intersections
    canopy_within = possible_matches[possible_matches.intersects(buffer_geom)]
    
    if len(canopy_within) == 0:
        return 0.0
    
    # Clip canopy to buffer and calculate area
    canopy_clipped = canopy_within.geometry.intersection(buffer_geom)
    canopy_area = canopy_clipped.area.sum()
    
    return canopy_area


print("\nCalculating canopy area for each buffer distance...")
print("This may take a few minutes for large datasets...")

# Initialize storage using actual dataframe index
n_properties = len(gdf_chc)
canopy_areas = {
    'canopy_0_20': pd.Series(index=gdf_chc.index, dtype=float),
    'canopy_0_50': pd.Series(index=gdf_chc.index, dtype=float),
    'canopy_0_100': pd.Series(index=gdf_chc.index, dtype=float),
    'canopy_0_200': pd.Series(index=gdf_chc.index, dtype=float)
}

# Calculate for each property
counter = 0
for idx, row in gdf_chc.iterrows():
    if counter % 1000 == 0:
        print(f"  Processing property {counter}/{n_properties}...")
    
    prop_geom = row.geometry
    
    # Calculate cumulative canopy areas - use idx directly
    canopy_areas['canopy_0_20'][idx] = calculate_canopy_in_buffer(
        prop_geom, row['buf_20'], canopy_chc, canopy_sindex
    )
    
    canopy_areas['canopy_0_50'][idx] = calculate_canopy_in_buffer(
        prop_geom, row['buf_50'], canopy_chc, canopy_sindex
    )
    
    canopy_areas['canopy_0_100'][idx] = calculate_canopy_in_buffer(
        prop_geom, row['buf_100'], canopy_chc, canopy_sindex
    )
    
    canopy_areas['canopy_0_200'][idx] = calculate_canopy_in_buffer(
        prop_geom, row['buf_200'], canopy_chc, canopy_sindex
    )
    
    counter += 1

print("✓ Canopy areas calculated")

# Add to dataframe
gdf_chc["canopy_0_20"]  = canopy_areas['canopy_0_20']
gdf_chc["canopy_0_50"]  = canopy_areas['canopy_0_50']
gdf_chc["canopy_0_100"] = canopy_areas['canopy_0_100']
gdf_chc["canopy_0_200"] = canopy_areas['canopy_0_200']

# ============================================================================
# STEP 4: CALCULATE RING AREAS (NOT CUMULATIVE)
# ============================================================================

print("\nCalculating buffer ring areas (donut rings)...")

# Calculate ring canopy areas by subtraction
gdf_chc["canopy_ring_0_20"]   = gdf_chc["canopy_0_20"]
gdf_chc["canopy_ring_20_50"]  = gdf_chc["canopy_0_50"] - gdf_chc["canopy_0_20"]
gdf_chc["canopy_ring_50_100"] = gdf_chc["canopy_0_100"] - gdf_chc["canopy_0_50"]
gdf_chc["canopy_ring_100_200"] = gdf_chc["canopy_0_200"] - gdf_chc["canopy_0_100"]

print("✓ Ring areas calculated")

# ============================================================================
# STEP 5: CALCULATE RING GEOMETRIC AREAS (FOR DENSITY CALCULATION)
# ============================================================================

print("\nCalculating geometric areas of each ring...")

# Ring geometric areas (in m²)
RING_AREA_0_20   = np.pi * (20**2)                    # π × 20²
RING_AREA_20_50  = np.pi * (50**2 - 20**2)           # π × (50² - 20²)
RING_AREA_50_100 = np.pi * (100**2 - 50**2)          # π × (100² - 50²)
RING_AREA_100_200 = np.pi * (200**2 - 100**2)        # π × (200² - 100²)

print(f"  Ring 0-20m:     {RING_AREA_0_20:,.0f} m²")
print(f"  Ring 20-50m:    {RING_AREA_20_50:,.0f} m²")
print(f"  Ring 50-100m:   {RING_AREA_50_100:,.0f} m²")
print(f"  Ring 100-200m:  {RING_AREA_100_200:,.0f} m²")

# ============================================================================
# STEP 6: CALCULATE CANOPY DENSITY (%) FOR EACH RING
# ============================================================================

print("\nCalculating canopy density (%) for each ring...")

# Calculate density as percentage
gdf_chc["canopy_density_0_20"] = (
    gdf_chc["canopy_ring_0_20"] / RING_AREA_0_20 * 100
)

gdf_chc["canopy_density_20_50"] = (
    gdf_chc["canopy_ring_20_50"] / RING_AREA_20_50 * 100
)

gdf_chc["canopy_density_50_100"] = (
    gdf_chc["canopy_ring_50_100"] / RING_AREA_50_100 * 100
)

gdf_chc["canopy_density_100_200"] = (
    gdf_chc["canopy_ring_100_200"] / RING_AREA_100_200 * 100
)

print("✓ Canopy densities calculated")

# ============================================================================
# STEP 7: SUMMARY STATISTICS
# ============================================================================

print("\n" + "="*80)
print("CANOPY DENSITY SUMMARY STATISTICS")
print("="*80)

density_cols = [
    'canopy_density_0_20',
    'canopy_density_20_50',
    'canopy_density_50_100',
    'canopy_density_100_200'
]

summary_stats = gdf_chc[density_cols].describe()
print("\n" + summary_stats.to_string())

print("\n" + "="*80)
print("CANOPY AREA SUMMARY STATISTICS (m²)")
print("="*80)

area_cols = [
    'canopy_ring_0_20',
    'canopy_ring_20_50',
    'canopy_ring_50_100',
    'canopy_ring_100_200'
]

area_stats = gdf_chc[area_cols].describe()
print("\n" + area_stats.to_string())

# ============================================================================
# STEP 8: DATA QUALITY CHECKS
# ============================================================================

print("\n" + "="*80)
print("DATA QUALITY CHECKS")
print("="*80)

print("\n1. Checking for negative values (should be none)...")
for col in density_cols:
    n_negative = (gdf_chc[col] < 0).sum()
    if n_negative > 0:
        print(f"   ⚠️  {col}: {n_negative} negative values found!")
    else:
        print(f"   ✓ {col}: No negative values")

print("\n2. Checking for values > 100% (should be none)...")
for col in density_cols:
    n_over_100 = (gdf_chc[col] > 100).sum()
    if n_over_100 > 0:
        print(f"   ⚠️  {col}: {n_over_100} values > 100%")
        print(f"      Max value: {gdf_chc[col].max():.2f}%")
    else:
        print(f"   ✓ {col}: All values ≤ 100%")

print("\n3. Checking for missing values...")
for col in density_cols:
    n_missing = gdf_chc[col].isna().sum()
    if n_missing > 0:
        print(f"   ⚠️  {col}: {n_missing} missing values")
    else:
        print(f"   ✓ {col}: No missing values")

# ============================================================================
# STEP 9: VISUALIZATION (OPTIONAL)
# ============================================================================

print("\n" + "="*80)
print("CREATING VISUALIZATIONS")
print("="*80)

import matplotlib.pyplot as plt
import seaborn as sns

# Box plots of canopy density by distance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Box plots
ax1 = axes[0]
gdf_chc[density_cols].boxplot(ax=ax1)
ax1.set_ylabel('Canopy Density (%)', fontsize=11)
ax1.set_xlabel('Distance Ring', fontsize=11)
ax1.set_title('Canopy Density Distribution by Distance Ring', fontweight='bold')
ax1.set_xticklabels(['0-20m', '20-50m', '50-100m', '100-200m'])
ax1.grid(True, alpha=0.3)

# Plot 2: Histograms
ax2 = axes[1]
for col in density_cols:
    label = col.replace('canopy_density_', '')
    ax2.hist(gdf_chc[col], bins=30, alpha=0.5, label=label)
ax2.set_xlabel('Canopy Density (%)', fontsize=11)
ax2.set_ylabel('Frequency', fontsize=11)
ax2.set_title('Distribution of Canopy Density', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('output/plots/canopy_rings/canopy_density_distribution.png', dpi=300, bbox_inches='tight')
print("✓ Saved: canopy_density_distribution.png")
plt.close()

# Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = gdf_chc[density_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, ax=ax,
            xticklabels=['0-20m', '20-50m', '50-100m', '100-200m'],
            yticklabels=['0-20m', '20-50m', '50-100m', '100-200m'])
ax.set_title('Correlation Matrix: Canopy Density by Distance', 
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('output/plots/canopy_rings/canopy_density_correlation.png', dpi=300, bbox_inches='tight')
print("✓ Saved: canopy_density_correlation.png")
plt.close()

# ============================================================================
# STEP 10: EXPORT RESULTS
# ============================================================================

print("\n" + "="*80)
print("EXPORTING RESULTS")
print("="*80)

# Select columns to export
export_cols = [
    'geometry',
    # Raw canopy areas (m²)
    'canopy_ring_0_20',
    'canopy_ring_20_50',
    'canopy_ring_50_100',
    'canopy_ring_100_200',
    # Canopy densities (%)
    'canopy_density_0_20',
    'canopy_density_20_50',
    'canopy_density_50_100',
    'canopy_density_100_200'
]

# Add existing columns if needed
existing_cols = [col for col in gdf_chc.columns if col not in export_cols]
export_cols_full = export_cols + existing_cols

# Export to GeoPackage
output_file = 'output/canopy_rings/property_with_canopy_density.gpkg'
# gdf_chc[export_cols_full].to_file(output_file, driver='GPKG')
# print(f"✓ Saved: {output_file}")

# Export summary to CSV
summary_df = pd.DataFrame({
    'Distance_Ring': ['0-20m', '20-50m', '50-100m', '100-200m'],
    'Ring_Area_m2': [RING_AREA_0_20, RING_AREA_20_50, RING_AREA_50_100, RING_AREA_100_200],
    'Mean_Canopy_Density_pct': [
        gdf_chc['canopy_density_0_20'].mean(),
        gdf_chc['canopy_density_20_50'].mean(),
        gdf_chc['canopy_density_50_100'].mean(),
        gdf_chc['canopy_density_100_200'].mean()
    ],
    'Median_Canopy_Density_pct': [
        gdf_chc['canopy_density_0_20'].median(),
        gdf_chc['canopy_density_20_50'].median(),
        gdf_chc['canopy_density_50_100'].median(),
        gdf_chc['canopy_density_100_200'].median()
    ],
    'Std_Canopy_Density_pct': [
        gdf_chc['canopy_density_0_20'].std(),
        gdf_chc['canopy_density_20_50'].std(),
        gdf_chc['canopy_density_50_100'].std(),
        gdf_chc['canopy_density_100_200'].std()
    ]
})

# summary_df.to_csv('canopy_density_summary.csv', index=False)
# print("✓ Saved: canopy_density_summary.csv")

print("\n" + "="*80)
print("COMPLETE!")
print("="*80)
print("\nFor hedonic models, use these variables:")
print("  • canopy_density_0_20   (canopy % within 20m)")
print("  • canopy_density_20_50  (canopy % in 20-50m ring)")
print("  • canopy_density_50_100 (canopy % in 50-100m ring)")
print("  • canopy_density_100_200 (canopy % in 100-200m ring)")
print("\nThese are directly comparable and interpretable!")
print("="*80)

Creating buffer rings...
✓ Buffers created

Building spatial index for canopy layer...
✓ Spatial index built

Calculating canopy area for each buffer distance...
This may take a few minutes for large datasets...
  Processing property 0/12431...
  Processing property 1000/12431...
  Processing property 2000/12431...
  Processing property 3000/12431...
  Processing property 4000/12431...
  Processing property 5000/12431...
  Processing property 6000/12431...
  Processing property 7000/12431...
  Processing property 8000/12431...
  Processing property 9000/12431...
  Processing property 10000/12431...
  Processing property 11000/12431...
  Processing property 12000/12431...
✓ Canopy areas calculated

Calculating buffer ring areas (donut rings)...
✓ Ring areas calculated

Calculating geometric areas of each ring...
  Ring 0-20m:     1,257 m²
  Ring 20-50m:    6,597 m²
  Ring 50-100m:   23,562 m²
  Ring 100-200m:  94,248 m²

Calculating canopy density (%) for each ring...
✓ Canopy densities

In [12]:
gdf_chc.columns

Index(['GrossSalePrice', 'AgeAtSale', 'LandArea', 'TotalFloorArea',
       'water_DIST', 'bus_DIST', 'Census_Pop', 'RnkIMDNoEm', 'RnkIMDNoIn',
       'RnkIMDNoCr', 'RnkIMDNoHo', 'RnkIMDNoHe', 'RnkIMDNoEd', 'RnkIMDNoAc',
       'DECILE_high', 'DECILE_prim', 'Median_Income', 'CBD_DIST', 'geometry',
       'cycleways_DIST', 'cycle_DENS', 'year_2018', 'year_2019', 'on_DIST',
       'on_DENS', 'buf_20', 'buf_50', 'buf_100', 'buf_200', 'canopy_0_20',
       'canopy_0_50', 'canopy_0_100', 'canopy_0_200', 'canopy_ring_0_20',
       'canopy_ring_20_50', 'canopy_ring_50_100', 'canopy_ring_100_200',
       'canopy_density_0_20', 'canopy_density_20_50', 'canopy_density_50_100',
       'canopy_density_100_200'],
      dtype='object')

In [13]:
gdf_chc.head()

,GrossSalePrice,AgeAtSale,LandArea,TotalFloorArea,water_DIST,bus_DIST,Census_Pop,RnkIMDNoEm,RnkIMDNoIn,RnkIMDNoCr,RnkIMDNoHo,RnkIMDNoHe,RnkIMDNoEd,RnkIMDNoAc,DECILE_high,DECILE_prim,Median_Income,CBD_DIST,geometry,cycleways_DIST,cycle_DENS,year_2018,year_2019,on_DIST,on_DENS,buf_20,buf_50,buf_100,buf_200,canopy_0_20,canopy_0_50,canopy_0_100,canopy_0_200,canopy_ring_0_20,canopy_ring_20_50,canopy_ring_50_100,canopy_ring_100_200,canopy_density_0_20,canopy_density_20_50,canopy_density_50_100,canopy_density_100_200
1560,610000,24,685.0,211.0,189.6017027471,216.7989101785,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6354.29138933796,POINT (1564919.129 5174157.305),590.0,1.6,False,True,338.1497192,2298.6,"POLYGON ((1564939.129 5174157.305, 1564939.033...","POLYGON ((1564969.129 5174157.305, 1564968.888...","POLYGON ((1565019.129 5174157.305, 1565018.647...","POLYGON ((1565119.129 5174157.305, 1565118.166...",1.077267,164.577741,1790.700463,12576.146645,1.077267,163.500474,1626.122721,10785.446182,0.085726,2.478277,6.901479,11.443714
3842,477500,42,609.0,120.0,139.5812001639,154.8247091108,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6227.39740786588,POINT (1565047.92 5174178.753),471.0710449,1.6,False,False,228.8069763,2303.7000000000003,"POLYGON ((1565067.92 5174178.753, 1565067.824 ...","POLYGON ((1565097.92 5174178.753, 1565097.679 ...","POLYGON ((1565147.92 5174178.753, 1565147.439 ...","POLYGON ((1565247.92 5174178.753, 1565246.957 ...",20.957896,471.372897,3086.451555,18773.967011,20.957896,450.415001,2615.078658,15687.515456,1.667776,6.827217,11.098739,16.644971
3671,655000,22,704.0,245.0,201.9424953009,192.9410290899,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6352.89184977953,POINT (1564904.931 5174179.536),610.8361206,1.6,True,False,358.6319275,2288.1,"POLYGON ((1564924.931 5174179.536, 1564924.835...","POLYGON ((1564954.931 5174179.536, 1564954.69 ...","POLYGON ((1565004.931 5174179.536, 1565004.449...","POLYGON ((1565104.931 5174179.536, 1565103.968...",10.299413,154.570187,2117.174622,11722.080865,10.299413,144.270775,1962.604434,9604.906243,0.819601,2.186801,8.329552,10.191122
3239,500000,44,869.0,131.0,157.8540143342,179.6454979671,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6229.0657185615,POINT (1565029.188 5174201.753),483.0162354,1.6,False,True,233.3314667,2296.0,"POLYGON ((1565049.188 5174201.753, 1565049.092...","POLYGON ((1565079.188 5174201.753, 1565078.947...","POLYGON ((1565129.188 5174201.753, 1565128.706...","POLYGON ((1565229.188 5174201.753, 1565228.225...",74.306779,467.130455,3099.836929,19298.350914,74.306779,392.823676,2632.706474,16198.513984,5.913146,5.954270,11.173553,17.187157
3781,880000,22,1149.0,279.0,199.6932187682,216.7306001091,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6258.81186920125,POINT (1564988.184 5174211.698),533.9364624,1.6,True,False,273.526001,2291.7,"POLYGON ((1565008.184 5174211.698, 1565008.087...","POLYGON ((1565038.184 5174211.698, 1565037.943...","POLYGON ((1565088.184 5174211.698, 1565087.702...","POLYGON ((1565188.184 5174211.698, 1565187.221...",0.000000,279.813206,2638.097872,17644.499905,0.000000,279.813206,2358.284666,15006.402033,0.000000,4.241300,10.008871,15.922287


In [14]:
gdf_chc.drop(inplace=True, columns=[
  'buf_20',
  'buf_50',
  'buf_100',
  'buf_200',
  'canopy_0_20',
  'canopy_0_50',
  'canopy_0_100',
  'canopy_0_200',
  # 'canopy_ring_0_20',
  # 'canopy_ring_20_50',
  # 'canopy_ring_50_100',
  # 'canopy_ring_100_200',
  # 'canopy_density_0_20',
  # 'canopy_density_20_50',
  # 'canopy_density_50_100',
  # 'canopy_density_100_200'
], errors='ignore')

In [15]:
gdf_chc.head()

,GrossSalePrice,AgeAtSale,LandArea,TotalFloorArea,water_DIST,bus_DIST,Census_Pop,RnkIMDNoEm,RnkIMDNoIn,RnkIMDNoCr,RnkIMDNoHo,RnkIMDNoHe,RnkIMDNoEd,RnkIMDNoAc,DECILE_high,DECILE_prim,Median_Income,CBD_DIST,geometry,cycleways_DIST,cycle_DENS,year_2018,year_2019,on_DIST,on_DENS,canopy_ring_0_20,canopy_ring_20_50,canopy_ring_50_100,canopy_ring_100_200,canopy_density_0_20,canopy_density_20_50,canopy_density_50_100,canopy_density_100_200
1560,610000,24,685.0,211.0,189.6017027471,216.7989101785,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6354.29138933796,POINT (1564919.129 5174157.305),590.0,1.6,False,True,338.1497192,2298.6,1.077267,163.500474,1626.122721,10785.446182,0.085726,2.478277,6.901479,11.443714
3842,477500,42,609.0,120.0,139.5812001639,154.8247091108,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6227.39740786588,POINT (1565047.92 5174178.753),471.0710449,1.6,False,False,228.8069763,2303.7000000000003,20.957896,450.415001,2615.078658,15687.515456,1.667776,6.827217,11.098739,16.644971
3671,655000,22,704.0,245.0,201.9424953009,192.9410290899,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6352.89184977953,POINT (1564904.931 5174179.536),610.8361206,1.6,True,False,358.6319275,2288.1,10.299413,144.270775,1962.604434,9604.906243,0.819601,2.186801,8.329552,10.191122
3239,500000,44,869.0,131.0,157.8540143342,179.6454979671,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6229.0657185615,POINT (1565029.188 5174201.753),483.0162354,1.6,False,True,233.3314667,2296.0,74.306779,392.823676,2632.706474,16198.513984,5.913146,5.954270,11.173553,17.187157
3781,880000,22,1149.0,279.0,199.6932187682,216.7306001091,930.0,1811.0,849.0,1231.0,1492.0,1516.0,1000.0,1327.0,7.0,10.0,79353,6258.81186920125,POINT (1564988.184 5174211.698),533.9364624,1.6,True,False,273.526001,2291.7,0.000000,279.813206,2358.284666,15006.402033,0.000000,4.241300,10.008871,15.922287


In [16]:
for col in gdf_chc.columns:
  if col != 'geometry':
    gdf_chc[col] = pd.to_numeric(gdf_chc[col], errors='coerce')
    
gdf_chc['year_2018'] = gdf_chc['year_2018'].astype(int)
gdf_chc['year_2019'] = gdf_chc['year_2019'].astype(int)

In [15]:
print(gdf_chc['canopy_0_20'].std())
print(gdf_chc['canopy_20_50'].std())
print(gdf_chc['GrossSalePrice'].mean())

108.50716096423331
481.32706251412293
546944.7268120022


In [17]:
gdf_chc.to_file('output/property_ring.gpkg')

In [18]:
gdf_chc['GrossSalePrice'].median()

np.float64(493000.0)